# Paper figures: the legacy glass-brain / cohort Grad-CAM plots

Ports the useful parts of `notebooks/legacy/3d-brain-plot.ipynb` (the paper's
figure-generation notebook, formerly `images/3D Brain Plot.ipynb`) onto the
real, committed root assets: `mri1.nii` (a real reference MRI volume) and the
derived Grad-CAM artifacts `pos_mean_mri.npy`, `pos_mean_pet.npy`,
`resized_pos_mri.npy`, `resized_neg_mri.npy` (see
`docs/legacy-notebooks-inventory.md`).

**Read every figure caption carefully**, this notebook mixes three different
kinds of image and it matters which is which:

1. **Real MRI reference** (`mri1.nii`): an actual structural MRI volume,
   shown both in its native orientation and after `process_scan` (the
   authoritative, ported preprocessing).
2. **Derived Grad-CAM maps** (the four `.npy` files): precomputed
   heatmaps from the original (unreproduced) training run, not raw scans
   and not generated by any code in this repository. `pos_mean_mri.npy`
   and `pos_mean_pet.npy` are cohort-mean maps; `resized_pos_mri.npy` and
   `resized_neg_mri.npy` are positive-/negative-class MRI Grad-CAM maps
   (see below for details on each).
3. **Static sample PNGs** (`samples/*.png`): pre-rendered figure crops from
   the paper, not reprocessed here at all.

The Grad-CAM-over-MRI overlays below reuse the legacy notebook's ad-hoc
affine nudge to line the heatmap up with the MRI slice **for illustration
only**. It is not a validated anatomical registration/coregistration, no
spatial-alignment claim should be drawn from it (same caveat as
`03-explainability.ipynb`'s atlas overlay). The glass-brain/stat-map
panels below use Nilearn's `plotting.plot_glass_brain`/`plot_stat_map`
directly, with the legacy notebook's exact parameters, reconstructed
against this same ad-hoc, unvalidated affine (Nilearn's world-coordinate
plotting does not make that affine any more anatomically correct, it
just renders it faithfully). Nilearn pulls in pandas transitively; that's
fine here (this repo's own tabular code stays Polars-only, see
AGENTS.md), pandas just isn't imported directly by anything in
`multimodal_ad` or this notebook.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
from nilearn import plotting

from multimodal_ad.data.manifest import Modality
from multimodal_ad.data.volumes import process_scan

REPO_ROOT = Path.cwd().parent  # notebook kernels cwd to notebooks/


## Load the real MRI reference and the derived Grad-CAM artifacts

`mri1.nii` is a real MRI volume; `process_scan` runs it through the same
ported preprocessing (`Dataset_MRI.ipynb`'s `process_scan`, see
`multimodal_ad.data.volumes`) used everywhere else in this repo, producing
the common `(128, 128, 50)` display shape. The four `.npy` arrays are
precomputed Grad-CAM outputs from the original training
run (not reproduced or regenerated here), already in that same shape.


In [ ]:
native_mri = nib.load(REPO_ROOT / "mri1.nii").get_fdata()
processed_mri = process_scan(REPO_ROOT / "mri1.nii", Modality.MRI)

pos_mean_mri = np.load(REPO_ROOT / "pos_mean_mri.npy")
pos_mean_pet = np.load(REPO_ROOT / "pos_mean_pet.npy")
resized_pos_mri = np.load(REPO_ROOT / "resized_pos_mri.npy")
resized_neg_mri = np.load(REPO_ROOT / "resized_neg_mri.npy")

native_mri.shape, processed_mri.shape, pos_mean_mri.shape, pos_mean_pet.shape


## Native MRI reference: orthogonal slices

Mid-volume sagittal/coronal/axial slices straight out of `mri1.nii`, in its
own native shape and orientation, before any of this repo's preprocessing.
This is the **real MRI reference volume**, not a derived heatmap.


In [ ]:
sx, sy, sz = (s // 2 for s in native_mri.shape)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(native_mri[sx, :, :].T, cmap="gray", origin="lower")
axes[0].set_title("Sagittal (native mri1.nii)")
axes[1].imshow(native_mri[:, sy, :].T, cmap="gray", origin="lower")
axes[1].set_title("Coronal (native mri1.nii)")
axes[2].imshow(native_mri[:, :, sz].T, cmap="gray", origin="lower")
axes[2].set_title("Axial (native mri1.nii)")
for ax in axes:
    ax.axis("off")
fig.suptitle("Real MRI reference, native orientation, no preprocessing")
fig.tight_layout()
plt.show()


## Processed MRI reference: central slice

The same volume after `process_scan` (normalize, 50 central axial frames,
brain-crop, resize to `128x128`), the shape every Grad-CAM overlay below is
aligned against.


In [ ]:
central = processed_mri.shape[-1] // 2

plt.figure(figsize=(4, 4))
plt.imshow(processed_mri[:, :, central], cmap="gray")
plt.title("Real MRI reference, processed (central frame)")
plt.axis("off")
plt.show()


## Derived Grad-CAM: positive vs. negative class, central slice

`resized_pos_mri` / `resized_neg_mri` are positive-/negative-class MRI
Grad-CAM heatmaps (already resized to `(128, 128, 50)`) from the
demented (positive) and non-demented (negative) class, respectively,
from the original training run, not regenerated here. Compared at the
same central-frame index as the processed MRI reference above.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(resized_pos_mri[:, :, central], cmap="magma")
axes[0].set_title("Derived Grad-CAM, positive class (central frame)")
axes[1].imshow(resized_neg_mri[:, :, central], cmap="magma")
axes[1].set_title("Derived Grad-CAM, negative class (central frame)")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()


## Cohort-mean Grad-CAM overlaid on the processed MRI reference

`pos_mean_mri` / `pos_mean_pet` are cohort-mean Grad-CAM heatmaps (positive
class), already in the same `(128, 128, 50)` shape as `processed_mri`, so
no legacy affine nudge is needed for this per-slice overlay (unlike the
glass-brain panels below, which reconstruct the legacy notebook's
world-coordinate framing). **Illustrative overlay only**: the MRI
reference and the two heatmaps come from different processing runs, this
is not a validated per-voxel correspondence.


In [ ]:
slice_indices = np.linspace(5, processed_mri.shape[-1] - 6, 4, dtype=int)

fig, axes = plt.subplots(2, len(slice_indices), figsize=(3 * len(slice_indices), 6))
for col, idx in enumerate(slice_indices):
    for row, (heatmap, label) in enumerate(
        [(pos_mean_mri, "MRI"), (pos_mean_pet, "PET")]
    ):
        ax = axes[row, col]
        ax.imshow(processed_mri[:, :, idx], cmap="gray")
        ax.imshow(heatmap[:, :, idx], cmap="inferno", alpha=0.5)
        ax.set_title(f"{label} mean Grad-CAM, frame {idx}")
        ax.axis("off")
fig.suptitle(
    "Cohort-mean Grad-CAM over processed MRI reference (illustrative overlay only)"
)
fig.tight_layout()
plt.show()


## Glass-brain and orthogonal statistical maps (Nilearn)

The legacy notebook used Nilearn's `plotting.plot_glass_brain`/`plot_stat_map` for this figure, with an ad-hoc affine assigned twice
in a row (the second assignment silently overrides the first, so only
the second one, `[[.8,0,0,17],[0,.9,0,-10],[0,0,.9,134],[0,0,0,1]]`
composed with `mri1.nii`'s own affine, was ever actually used to render
the paper's figures). Reproduced here with the same library, the same
affine, and the same plotting parameters (`display_mode`, `threshold`,
`cmap`, `cut_coords`, `colorbar`). **The affine itself is still ad hoc
and unvalidated** (see the note at the top of this notebook and
`docs/legacy-notebooks-inventory.md`); reproducing the exact plotting
call does not validate the underlying spatial alignment.


In [ ]:
effective_affine = np.array(
    [
        [0.8, 0, 0, 17],
        [0, 0.9, 0, -10],
        [0, 0, 0.9, 134],
        [0, 0, 0, 1],
    ]
) @ nib.load(REPO_ROOT / "mri1.nii").affine

nii_mri = nib.Nifti2Image(pos_mean_mri, affine=effective_affine)
nii_pet = nib.Nifti2Image(pos_mean_pet, affine=effective_affine)

for nii, title in [(nii_mri, "MRI"), (nii_pet, "PET")]:
    plotting.plot_glass_brain(
        nii,
        display_mode="ortho",
        threshold="auto",
        cmap="jet",
        cut_coords=range(0, 51, 10),
        colorbar=True,
        title=f"{title} mean Grad-CAM, glass brain",
    )
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(20, 10))
for nii, title, ax in [(nii_mri, "MRI", axes[0]), (nii_pet, "PET", axes[1])]:
    # list(...), not range(...): this Nilearn version rejects a bare
    # range for a single-axis display_mode; same coordinate values.
    plotting.plot_stat_map(
        nii,
        display_mode="z",
        threshold="auto",
        cmap="jet",
        cut_coords=list(range(0, 51, 10)),
        colorbar=True,
        axes=ax,
    )
    ax.set_title(title)
plt.show()


## Static sample PNGs

`samples/mri-sample.png` and `samples/pet-sample.png` are pre-rendered
figure crops committed alongside the paper. They are **not** reprocessed
or regenerated by any code in this notebook or this repository, shown here
only for reference alongside the reproduced figures above.


In [ ]:
mri_sample = plt.imread(REPO_ROOT / "samples" / "mri-sample.png")
pet_sample = plt.imread(REPO_ROOT / "samples" / "pet-sample.png")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(mri_sample)
axes[0].set_title("Static sample PNG: MRI (paper figure crop)")
axes[1].imshow(pet_sample)
axes[1].set_title("Static sample PNG: PET (paper figure crop)")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()


## Next steps

- Legacy figure-generation notebook: `notebooks/legacy/3d-brain-plot.ipynb`.
- Spatial-alignment caveats (no registration step) are documented in
  `docs/legacy-notebooks-inventory.md` and in `multimodal_ad.models.regions`
  (`ATLAS_PLACEMENT`/`HEATMAP_PLACEMENT`), which this notebook's overlays
  echo but do not resolve.
